# G1 Humanoid Robot LQR Balancing Control

This notebook implements Time-Varying Linear Quadratic Regulator (TVLQR) control for balancing the G1 humanoid robot using:
- GTDynamics for trajectory optimization and LQR policy computation
- MuJoCo for physics simulation
- Matplotlib (pyplot) for visualization

## Workflow:
1. Load G1 robot model
2. Build factor graph for balancing optimization
3. Optimize trajectory (standing still)
4. Linearize around trajectory to get Gaussian factor graph
5. Eliminate to get Bayes net (LQR policy)
6. Simulate with MuJoCo applying TVLQR feedback control
7. Visualize results

## 1. Imports and Setup

In [1]:
import numpy as np
import gtsam
import gtdynamics as gtd
import mujoco
import mujoco.viewer
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import mediapy as media

print("Imports successful!")
print(f"GTDynamics version: {gtd.__version__ if hasattr(gtd, '__version__') else 'Unknown'}")
print(f"GTSAM version: {gtsam.__version__ if hasattr(gtsam, '__version__') else 'Unknown'}")
print(f"MuJoCo version: {mujoco.__version__ if hasattr(mujoco, '__version__') else 'Unknown'}")

Imports successful!
GTDynamics version: Unknown
GTSAM version: Unknown
MuJoCo version: 3.3.2


## 2. MuJoCo Viewer Playground
Interactive viewer to explore the G1 robot model before running optimization.

In [2]:
# MuJoCo Viewer Playground - Only run for interactive exploration
MJCF_PATH = '../../models/urdfs/g1_description/g1_23dof.xml'

model = mujoco.MjModel.from_xml_path(MJCF_PATH)
data = mujoco.MjData(model)

#mujoco.viewer.launch(model, data)

## 3. Load G1 Robot Model

In [3]:
# Load G1 robot from URDF
URDF_PATH = '../../models/urdfs/g1_description/g1_23dof.urdf'
robot = gtd.CreateRobotFromFile(URDF_PATH)

print(f"Number of links: {robot.numLinks()}")
print(f"Number of joints: {robot.numJoints()}")

# Print link names
print("\nLink names:")
for i, link in enumerate(robot.links()):
    print(f"{link.name()}: {link.id()}")


# Print joint names
print("\nJoint names:")
for i, joint in enumerate(robot.joints()):
    print(f"  {i}: {joint.name()}")

Number of links: 24
Number of joints: 23

Link names:
left_ankle_pitch_link: 5
left_ankle_roll_link: 6
left_elbow_link: 17
left_hip_pitch_link: 1
left_hip_roll_link: 2
left_hip_yaw_link: 3
left_knee_link: 4
left_shoulder_pitch_link: 14
left_shoulder_roll_link: 15
left_shoulder_yaw_link: 16
left_wrist_roll_rubber_hand: 18
pelvis: 0
right_ankle_pitch_link: 11
right_ankle_roll_link: 12
right_elbow_link: 22
right_hip_pitch_link: 7
right_hip_roll_link: 8
right_hip_yaw_link: 9
right_knee_link: 10
right_shoulder_pitch_link: 19
right_shoulder_roll_link: 20
right_shoulder_yaw_link: 21
right_wrist_roll_rubber_hand: 23
torso_link: 13

Joint names:
  0: left_ankle_pitch_joint
  1: left_ankle_roll_joint
  2: left_elbow_joint
  3: left_hip_pitch_joint
  4: left_hip_roll_joint
  5: left_hip_yaw_joint
  6: left_knee_joint
  7: left_shoulder_pitch_joint
  8: left_shoulder_roll_joint
  9: left_shoulder_yaw_joint
  10: left_wrist_roll_joint
  11: right_ankle_pitch_joint
  12: right_ankle_roll_joint
  13:

## 4. Trajectory Optimization Parameters

In [4]:
# Time parameters
balance_time = 3.0  # seconds - duration of balancing
balance_steps = 300  # number of time steps
balance_dt = balance_time / balance_steps  # time step duration

print(f"Optimization horizon: {balance_time} seconds")
print(f"Number of time steps: {balance_steps}")
print(f"Time step size (dt): {balance_dt:.4f} seconds")

# Get number of joints
pelvis_link_id = 0
num_links = robot.numLinks()
num_joints = robot.numJoints()

# Initial configuration - G1 standing pose (neutral position)
# All joints start at zero (neutral standing pose)
initial_joint_angles = np.zeros(num_joints)

print(f"\nInitial joint configuration set for {num_joints} joints (all zero - neutral pose)")

Optimization horizon: 3.0 seconds
Number of time steps: 300
Time step size (dt): 0.0100 seconds

Initial joint configuration set for 23 joints (all zero - neutral pose)


## 4.5 Define Contact Points

We'll define contact points at both feet to ensure proper ground contact during balancing.

In [5]:
# Define contact points at the feet
# We need to identify which links are the feet and define contact points

left_foot = robot.link('left_ankle_roll_link')
right_foot = robot.link('right_ankle_roll_link')
foot_links = [left_foot, right_foot]

contacts_in_link_frame = [
    np.array([-0.05, 0.025, -0.035]),
    np.array([-0.05, -0.025, -0.035]),
    np.array([0.12, 0.03, -0.035]),
    np.array([0.12, -0.03, -0.035])
]

# Create list of contact points
contact_points = []
for link in foot_links:
    for contact_in_link_frame in contacts_in_link_frame:
        contact_points.append(gtd.PointOnLink(link, contact_in_link_frame))
        print(f"Added contact point for {link.name()} at {contact_in_link_frame}")

print(f"\nTotal contact points: {len(contact_points)}")
ground_plane_height = 0.0  # Ground at z=0
friction_coefficient = 1.0  # Friction coefficient for contact

Added contact point for left_ankle_roll_link at [-0.05   0.025 -0.035]
Added contact point for left_ankle_roll_link at [-0.05  -0.025 -0.035]
Added contact point for left_ankle_roll_link at [ 0.12   0.03  -0.035]
Added contact point for left_ankle_roll_link at [ 0.12  -0.03  -0.035]
Added contact point for right_ankle_roll_link at [-0.05   0.025 -0.035]
Added contact point for right_ankle_roll_link at [-0.05  -0.025 -0.035]
Added contact point for right_ankle_roll_link at [ 0.12   0.03  -0.035]
Added contact point for right_ankle_roll_link at [ 0.12  -0.03  -0.035]

Total contact points: 8


## 5. Build Factor Graph for Balancing

In [26]:
# Cost model parameters (noise models)
sigma_dynamics = 1e-4      
sigma_objectives = 1e-3    
sigma_torque = 1e-0        

# Create noise models
dynamics_model_1 = gtsam.noiseModel.Isotropic.Sigma(1, sigma_dynamics)
dynamics_model_3 = gtsam.noiseModel.Isotropic.Sigma(3, sigma_dynamics)
dynamics_model_6 = gtsam.noiseModel.Isotropic.Sigma(6, sigma_dynamics)
objectives_model_1 = gtsam.noiseModel.Isotropic.Sigma(1, sigma_objectives)
objectives_model_6 = gtsam.noiseModel.Isotropic.Sigma(6, sigma_objectives)
soft_objectives_1 = gtsam.noiseModel.Isotropic.Sigma(1, 1e-2)
torque_model = gtsam.noiseModel.Isotropic.Sigma(1, sigma_torque)
base_model_6 = gtsam.noiseModel.Isotropic.Sigma(6, sigma_objectives)  # Soft base constraints

# Create optimization settings and graph builder
gravity_vec = np.array([0, 0, -9.81])
opt_params = gtd.OptimizerSetting(sigma_dynamics, sigma_contact=1e1)
print(opt_params.cm_cost_model)
graph_builder = gtd.DynamicsGraph(opt_params, gravity_vec, None)

# Build the trajectory factor graph WITH contact points
# This automatically adds dynamics, kinematics, collocation constraints, AND contact constraints
graph = graph_builder.trajectoryFG(
    robot, 
    balance_steps, 
    balance_dt,
    gtd.CollocationScheme.Trapezoidal,  # Collocation scheme
    contact_points,  # Contact points at feet
    friction_coefficient  # Friction coefficient
)

print(f"Factor graph built with {graph.size()} factors (including contact constraints)")

isotropic dim=3 sigma=10

Factor graph built with 67679 factors (including contact constraints)


## 5.1 Forward Kinematics

In [7]:
# Use FK with left ankle roll link as the base and all joints at 0

fk_values = gtsam.Values()

# Add all joint angles at 0
for j in range(robot.numJoints()):
    fk_values.insert(gtd.JointAngleKey(j, 0), 0.0)

pelvis_link = robot.link('pelvis')
left_ankle_link = robot.link('left_ankle_roll_link')
right_ankle_link = robot.link('right_ankle_roll_link')

# Place left ankle at the specified position from URDF
# Position: [-2.3261e-06, 0.118506, -0.756864]
# Rotation: Identity (no rotation)
left_ankle_pose = gtsam.Pose3(
    gtsam.Rot3(),  # Identity rotation
    np.array([0.0, 0.0, 0.035])
)
fk_values.insert(gtd.PoseKey(left_ankle_link.id(), 0), left_ankle_pose)

# Compute forward kinematics using left ankle as the prior link
fk_result = robot.forwardKinematics(fk_values, 0, left_ankle_link.name())

print("Forward Kinematics Results (all joints at 0, left_ankle_roll_link as base):")
print("=" * 80)

# Print left ankle position (should match what we set)
print(f"\n{left_ankle_link.name()} (id: {left_ankle_link.id()}) [BASE/PRIOR LINK]:")
left_ankle_result = gtd.Pose(fk_result, left_ankle_link.id(), 0)
left_ankle_pos = left_ankle_result.translation()
print(f"  Position: {left_ankle_pos}")
print(f"  z-coordinate: {left_ankle_pos[2]:.6f} m")

# Print right ankle position (should be at specified position)
if right_ankle_link:
    print(f"\n{right_ankle_link.name()} (id: {right_ankle_link.id()}):")
    right_ankle_result = gtd.Pose(fk_result, right_ankle_link.id(), 0)
    right_ankle_pos = right_ankle_result.translation()
    print(f"  Position: {right_ankle_pos}")
    print(f"  z-coordinate: {right_ankle_pos[2]:.6f} m")

# Print pelvis position (should be derived from FK)
print(f"\npelvis (id: {pelvis_link.id()}):")
pelvis_pose = gtd.Pose(fk_result, pelvis_link.id(), 0)
pelvis_pos = pelvis_pose.translation()
print(f"  Position: {pelvis_pos}")
print(f"  z-coordinate: {pelvis_pos[2]:.6f} m")

# Print all link positions
print("\n" + "=" * 80)
print("All Link Positions:")
print("=" * 80)
for link in robot.links():
    link_pose = gtd.Pose(fk_result, link.id(), 0)
    link_pos = link_pose.translation()
    print(f"{link.id():3d}: {link.name():30s} - Position: [{link_pos[0]:10.6f}, {link_pos[1]:10.6f}, {link_pos[2]:10.6f}]")

Forward Kinematics Results (all joints at 0, left_ankle_roll_link as base):

left_ankle_roll_link (id: 6) [BASE/PRIOR LINK]:
  Position: [0.    0.    0.035]
  z-coordinate: 0.035000 m

right_ankle_roll_link (id: 12):
  Position: [ 0.         -0.23701291  0.035     ]
  z-coordinate: 0.035000 m

pelvis (id: 0):
  Position: [-0.02650267 -0.11850645  0.73225869]
  z-coordinate: 0.732259 m

All Link Positions:
  5: left_ankle_pitch_link          - Position: [ -0.033774,   0.000000,   0.080120]
  6: left_ankle_roll_link           - Position: [  0.000000,   0.000000,   0.035000]
 17: left_elbow_link                - Position: [  0.054228,   0.032744,   0.903466]
  1: left_hip_pitch_link            - Position: [ -0.023762,  -0.006263,   0.679529]
  2: left_hip_roll_link             - Position: [  0.018156,  -0.003099,   0.593719]
  3: left_hip_yaw_link              - Position: [ -0.010876,  -0.013035,   0.398726]
  4: left_knee_link                 - Position: [ -0.021048,   0.004058,   0.2482

In [8]:
# Use FK with pelvis as the base (using pose from the previous cell) and all joints at 0
fk_values = gtsam.Values()

# Add all joint angles at 0
for j in range(robot.numJoints()):
    fk_values.insert(gtd.JointAngleKey(j, 0), 0.0)

# Place left ankle at the specified position from URDF
# Position: [-2.3261e-06, 0.118506, -0.756864]
# Rotation: Identity (no rotation)
pelvis_link_pose = gtsam.Pose3(
    gtsam.Rot3(),  # Identity rotation
    np.array([0.0, 0.0, 0.73225869])
)
fk_values.insert(gtd.PoseKey(pelvis_link.id(), 0), pelvis_link_pose)

# Compute forward kinematics using left ankle as the prior link
fk_result = robot.forwardKinematics(fk_values, 0, "pelvis")

print("Forward Kinematics Results (all joints at 0, pelvis as base):")
print("=" * 80)

# Print left ankle position (should match what we set)
print(f"\n{"pelvis"} (id: {pelvis_link.id()}) [BASE/PRIOR LINK]:")
pelvis_result = gtd.Pose(fk_result, pelvis_link.id(), 0)
pelvis_pose = pelvis_result.translation()
print(f"  Position: {pelvis_pose}")
print(f"  z-coordinate: {pelvis_pose[2]:.6f} m")

# Print right ankle position (should be at specified position)
if right_ankle_link:
    print(f"\n{right_ankle_link.name()} (id: {right_ankle_link.id()}):")
    right_ankle_result = gtd.Pose(fk_result, right_ankle_link.id(), 0)
    right_ankle_pos = right_ankle_result.translation()
    print(f"  Position: {right_ankle_pos}")
    print(f"  z-coordinate: {right_ankle_pos[2]:.6f} m")

# Print pelvis position (should be derived from FK)
print(f"\left ankle (id: {left_ankle_link.id()}):")
left_ankle_pose = gtd.Pose(fk_result, left_ankle_link.id(), 0)
left_ankle_pos = left_ankle_pose.translation()
print(f"  Position: {left_ankle_pos}")
print(f"  z-coordinate: {left_ankle_pos[2]:.6f} m")

# Print all link positions
print("\n" + "=" * 80)
print("All Link Positions:")
print("=" * 80)
for link in robot.links():
    link_pose = gtd.Pose(fk_result, link.id(), 0)
    link_pos = link_pose.translation()
    print(f"{link.id():3d}: {link.name():30s} - Position: [{link_pos[0]:10.6f}, {link_pos[1]:10.6f}, {link_pos[2]:10.6f}]")

Forward Kinematics Results (all joints at 0, pelvis as base):

pelvis (id: 0) [BASE/PRIOR LINK]:
  Position: [0.         0.         0.73225869]
  z-coordinate: 0.732259 m

right_ankle_roll_link (id: 12):
  Position: [ 0.02650267 -0.11850645  0.035     ]
  z-coordinate: 0.035000 m
\left ankle (id: 6):
  Position: [0.02650267 0.11850645 0.035     ]
  z-coordinate: 0.035000 m

All Link Positions:
  5: left_ankle_pitch_link          - Position: [ -0.007271,   0.118506,   0.080120]
  6: left_ankle_roll_link           - Position: [  0.026503,   0.118506,   0.035000]
 17: left_elbow_link                - Position: [  0.080731,   0.151251,   0.903466]
  1: left_hip_pitch_link            - Position: [  0.002741,   0.112243,   0.679529]
  2: left_hip_roll_link             - Position: [  0.044659,   0.115407,   0.593719]
  3: left_hip_yaw_link              - Position: [  0.015626,   0.105471,   0.398726]
  4: left_knee_link                 - Position: [  0.005455,   0.122565,   0.248253]
 14: lef

<>:39: SyntaxWarning: invalid escape sequence '\l'
<>:39: SyntaxWarning: invalid escape sequence '\l'
/tmp/ipykernel_63583/3668440332.py:39: SyntaxWarning: invalid escape sequence '\l'
  print(f"\left ankle (id: {left_ankle_link.id()}):")


## 6. Add Boundary Conditions and Objectives

In [9]:
# Initial state: standing configuration with zero velocities
for j in range(num_joints):
    # Initial joint angles
    graph.addPriorDouble(gtd.JointAngleKey(j, 0), initial_joint_angles[j], dynamics_model_1)
    # Initial velocities (zero - standing still)
    graph.addPriorDouble(gtd.JointVelKey(j, 0), 0.0, dynamics_model_1)

# Constrain pelvis using pose from FK above
desired_base_pose = gtsam.Pose3(gtsam.Rot3(), np.array([0.0, 0.0, 0.73225869]))
pose_prior = gtsam.PriorFactorPose3(gtd.PoseKey(pelvis_link_id, 0), desired_base_pose, dynamics_model_6)
graph.add(pose_prior)

# Add base twist constraint
zero_twist = np.zeros(6)
twist_prior = gtsam.PriorFactorVector(gtd.TwistKey(pelvis_link_id, 0), zero_twist, dynamics_model_6)
graph.add(twist_prior)

In [10]:
# For all intermediate time steps - objectives to stay at standing configuration
for t in range(1, balance_steps):
    graph.add(gtsam.PriorFactorPose3(gtd.PoseKey(pelvis_link_id, t), desired_base_pose, objectives_model_6))
    
    # Add twist objective - stay at zero velocity
    twist_objective = gtsam.PriorFactorVector(gtd.TwistKey(pelvis_link_id, t), zero_twist, objectives_model_6)
    graph.add(twist_objective)

    for j in range(num_joints):
        # Angle objectives - maintain standing pose
        graph.addPriorDouble(gtd.JointAngleKey(j, t), initial_joint_angles[j], soft_objectives_1)
        # Velocity objectives - stay still
        graph.addPriorDouble(gtd.JointVelKey(j, t), 0.0, soft_objectives_1)
        # Acceleration objectives - stay still
        graph.addPriorDouble(gtd.JointAccelKey(j, t), 0.0, soft_objectives_1)

In [11]:
# Final state pelvis pose constraint (same as initial)
graph.add(gtsam.PriorFactorPose3(gtd.PoseKey(pelvis_link_id, balance_steps), desired_base_pose, objectives_model_6))

# Add final twist constraint (zero velocity - standing still)
twist_final = gtsam.PriorFactorVector(gtd.TwistKey(pelvis_link_id, balance_steps), zero_twist, objectives_model_6)
graph.add(twist_final)
    
for j in range(num_joints):
    graph.addPriorDouble(gtd.JointAngleKey(j, balance_steps), initial_joint_angles[j], soft_objectives_1)
    graph.addPriorDouble(gtd.JointVelKey(j, balance_steps), 0.0, soft_objectives_1)
    graph.addPriorDouble(gtd.JointAccelKey(j, balance_steps), 0.0, soft_objectives_1)

In [12]:
# Add minimum torque objectives (encourage small control effort)
for t in range(balance_steps + 1):
    for j in range(num_joints):
        graph.add(gtd.MinTorqueFactor(gtd.TorqueKey(j, t), torque_model))

In [13]:
print(f"Added boundary conditions and torque minimization")
print(f"Total factors in graph: {graph.size()}")

Added boundary conditions and torque minimization
Total factors in graph: 95950


## 7. Initialize Values and Optimize

In [14]:
# Initialize values using GTDynamics Initializer with contact points
initializer = gtd.Initializer()
init_values = initializer.ZeroValuesTrajectory(robot, balance_steps, 0, 1e-5, contact_points)

# For each timestep, use FK to get consistent link poses
for t in range(balance_steps + 1):
    # Create temporary values for this timestep
    fk_values_t = gtsam.Values()
    
    # Set all joint angles to zero (standing pose)
    for j in range(num_joints):
        fk_values_t.insert(gtd.JointAngleKey(j, t), 0.0)
    
    # Set left ankle COM at ground level
    pelvis_link_pose = gtsam.Pose3(gtsam.Rot3(), np.array([0.0, 0.0, 0.73225869])) # Pose from FK
    fk_values_t.insert(gtd.PoseKey(pelvis_link.id(), t), pelvis_link_pose)
    
    # Run FK with left ankle as prior
    fk_result = robot.forwardKinematics(fk_values_t, t, pelvis_link.name())
    
    # Copy FK results to init_values for all links
    for link in robot.links():
        pose_key = gtd.PoseKey(link.id(), t)
        if init_values.exists(pose_key):
            init_values.erase(pose_key)
        init_values.insert(pose_key, gtd.Pose(fk_result, link.id(), t))

In [15]:
# The twist priors use PriorFactorVector which expects VectorX (dynamic size)
# But ZeroValuesTrajectory inserts Vector6 (fixed size)
# We need to update all twist values to be stored as VectorX
zero_twist_vectorx = np.zeros(6)  # This will be stored as dynamic vector

# Update twist values for all timesteps
for t in range(balance_steps + 1):
    for link in robot.links():
        twist_key = gtd.TwistKey(link.id(), t)
        twist_accel_key = gtd.TwistAccelKey(link.id(), t)
        if init_values.exists(twist_key):
            init_values.erase(twist_key)
            init_values.insert(twist_key, zero_twist_vectorx)
        if init_values.exists(twist_accel_key):
            init_values.erase(twist_accel_key)
            init_values.insert(twist_accel_key, zero_twist_vectorx)


In [16]:
print(f"Initialized {init_values.size()} variables with FK-consistent poses")
print(f"Pelvis height at t=0: {gtd.Pose(init_values, pelvis_link_id, 0).translation()[2]:.3f}m")
print(f"Left ankle height at t=0: {gtd.Pose(init_values, left_ankle_link.id(), 0).translation()[2]:.3f}m")
print(f"Right ankle height at t=0: {gtd.Pose(init_values, right_ankle_link.id(), 0).translation()[2]:.3f}m")

Initialized 65618 variables with FK-consistent poses
Pelvis height at t=0: 0.732m
Left ankle height at t=0: 0.035m
Right ankle height at t=0: 0.035m


## 7.1 Analyze Initial Error Before Optimization

Before running the optimization, let's analyze the factor graph errors on the initial zero values to understand the starting point of the problem.

In [17]:
# Analyze factor contributions to INITIAL error (before optimization)
print("Analyzing factor contributions to INITIAL error (before optimization)...")
print(f"Initial total error: {graph.error(init_values):.6e}\n")

# Get all factors and their errors on initial values
initial_factor_errors = []
initial_factor_types = []

for i in range(graph.size()):
    factor = graph.at(i)
    error = factor.error(init_values)
    initial_factor_errors.append(error)
    
    # Get the actual factor type name using Python's type system
    factor_type = type(factor).__name__
    initial_factor_types.append(factor_type)

# Convert to numpy arrays
initial_factor_errors = np.array(initial_factor_errors)
initial_factor_types = np.array(initial_factor_types)

# Group by factor type
unique_types = np.unique(initial_factor_types)
type_errors = {}
type_counts = {}

for ftype in unique_types:
    mask = initial_factor_types == ftype
    type_errors[ftype] = np.sum(initial_factor_errors[mask])
    type_counts[ftype] = np.sum(mask)

# Sort by error contribution
sorted_types = sorted(type_errors.keys(), key=lambda x: type_errors[x], reverse=True)

print(f"Factor Error Analysis (Initial Values):")
print(f"Total factors: {len(initial_factor_errors)}")
print(f"Total error: {np.sum(initial_factor_errors):.6e}")
print(f"\nError by factor type:")
for ftype in sorted_types:
    print(f"  {ftype:40s}: {type_errors[ftype]:12.6e} ({type_counts[ftype]:4d} factors, avg: {type_errors[ftype]/type_counts[ftype]:.6e})")

# Create visualization with Plotly
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Total Error by Factor Type (Initial)',
        'Number of Factors by Type',
        'Average Error per Factor by Type (Initial)',
        'Error Distribution (Log Scale)'
    ),
    specs=[[{"type": "bar"}, {"type": "bar"}],
           [{"type": "bar"}, {"type": "histogram"}]]
)

# Plot 1: Total error by factor type
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#bcbd22', '#17becf']
fig.add_trace(
    go.Bar(x=sorted_types, y=[type_errors[t] for t in sorted_types],
           marker_color=colors[:len(sorted_types)],
           name='Total Error'),
    row=1, col=1
)

# Plot 2: Number of factors by type
fig.add_trace(
    go.Bar(x=sorted_types, y=[type_counts[t] for t in sorted_types],
           marker_color=colors[:len(sorted_types)],
           name='Count'),
    row=1, col=2
)

# Plot 3: Average error per factor by type
fig.add_trace(
    go.Bar(x=sorted_types, y=[type_errors[t]/type_counts[t] for t in sorted_types],
           marker_color=colors[:len(sorted_types)],
           name='Avg Error'),
    row=2, col=1
)

# Plot 4: Error distribution histogram (log scale)
fig.add_trace(
    go.Histogram(x=np.log10(initial_factor_errors[initial_factor_errors > 0]),
                 nbinsx=50,
                 marker_color='#1f77b4',
                 name='Error Distribution'),
    row=2, col=2
)

# Update layout
fig.update_xaxes(tickangle=45, row=1, col=1)
fig.update_xaxes(tickangle=45, row=1, col=2)
fig.update_xaxes(tickangle=45, row=2, col=1)
fig.update_xaxes(title_text="Log10(Error)", row=2, col=2)

fig.update_yaxes(type="log", title_text="Total Error", row=1, col=1)
fig.update_yaxes(title_text="Number of Factors", row=1, col=2)
fig.update_yaxes(type="log", title_text="Average Error", row=2, col=1)
fig.update_yaxes(title_text="Count", row=2, col=2)

fig.update_layout(
    height=800,
    title_text="Factor Error Analysis (Initial Values - Before Optimization)",
    showlegend=False
)

fig.show()

Analyzing factor contributions to INITIAL error (before optimization)...
Initial total error: 1.993457e+14

Factor Error Analysis (Initial Values):
Total factors: 95950
Total error: 1.993457e+14

Error by factor type:
  NonlinearFactor                         : 1.993457e+14 (65271 factors, avg: 3.054124e+09)
  PriorFactorDouble                       : 1.993382e-01 (20746 factors, avg: 9.608515e-06)
  MinTorqueFactor                         : 2.262501e-07 (6923 factors, avg: 3.268093e-11)
  ContactHeightFactor                     : 5.401402e-17 (2408 factors, avg: 2.243107e-20)
  PriorFactorPose3                        : 0.000000e+00 ( 301 factors, avg: 0.000000e+00)
  PriorFactorVector                       : 0.000000e+00 ( 301 factors, avg: 0.000000e+00)


In [18]:
# Print top factors with largest INITIAL errors
number_of_factors = 30
print("\n" + "="*80)
print(f"TOP {number_of_factors} FACTORS WITH LARGEST INITIAL ERRORS")
print("="*80)

# Create list of (error, factor_type, factor_index) tuples
initial_factor_info = [(initial_factor_errors[i], initial_factor_types[i], i) for i in range(len(initial_factor_errors))]

# Sort by error in descending order
initial_factor_info_sorted = sorted(initial_factor_info, key=lambda x: x[0], reverse=True)

# Print top factors
print(f"\n{'Rank':<6} {'Error':<15} {'Factor Type':<40} {'Factor Index':<12}")
print("-" * 80)
for rank, (error, ftype, idx) in enumerate(initial_factor_info_sorted[:number_of_factors], 1):
    # Get the keys associated with this factor
    factor = graph.at(idx)
    keys = factor.keys()
    key_str = ", ".join([gtd.GTDKeyFormatter(k) for k in keys])
    
    print(f"{rank:<6} {error:<15.6e} {ftype:<40} {idx:<12}")
    print(f"       Keys: {key_str}")
    print()

print("="*80)


TOP 30 FACTORS WITH LARGEST INITIAL ERRORS

Rank   Error           Factor Type                              Factor Index
--------------------------------------------------------------------------------
1      4.662848e+11    NonlinearFactor                          132         
       Keys: A[13]0, F[13](12)0, F[13](13)0, F[13](18)0, V[13]0, p[13]0

2      4.662848e+11    NonlinearFactor                          357         
       Keys: A[13]1, F[13](12)1, F[13](13)1, F[13](18)1, V[13]1, p[13]1

3      4.662848e+11    NonlinearFactor                          582         
       Keys: A[13]2, F[13](12)2, F[13](13)2, F[13](18)2, V[13]2, p[13]2

4      4.662848e+11    NonlinearFactor                          807         
       Keys: A[13]3, F[13](12)3, F[13](13)3, F[13](18)3, V[13]3, p[13]3

5      4.662848e+11    NonlinearFactor                          1032        
       Keys: A[13]4, F[13](12)4, F[13](13)4, F[13](18)4, V[13]4, p[13]4

6      4.662848e+11    NonlinearFactor         

In [19]:
# Check contact initialization
print("\nContact Point Initialization Check:")
print("="*80)

# Group contacts by link to match C++ implementation
contacts_by_link = {}
for cp in contact_points:
    link_id = cp.link.id()
    if link_id not in contacts_by_link:
        contacts_by_link[link_id] = []
    contacts_by_link[link_id].append(cp)

# Check each contact with proper contact IDs
for link_id, link_contacts in contacts_by_link.items():
    for contact_id, cp in enumerate(link_contacts):
        link = cp.link
        offset = cp.point
        print(f"\nContact on {link.name()} (contact_id={contact_id}):")
        print(f"  Link COM pose at t=0: {gtd.Pose(init_values, link.id(), 0).translation()}")
        print(f"  Contact offset in link frame: {offset}")
        
        # Compute contact point in world frame
        link_pose = gtd.Pose(init_values, link.id(), 0)
        contact_world = link_pose.transformFrom(offset)
        print(f"  Contact point in world: {contact_world}")
        print(f"  Expected z=0, actual z={contact_world[2]:.6f}")
        
        # Check if wrench is initialized with proper contact_id
        wrench_key = gtd.ContactWrenchKey(link.id(), contact_id, 0)
        if init_values.exists(wrench_key):
            wrench = init_values.atVector(wrench_key)
            print(f"  Contact wrench (ContactWrenchKey({link.id()}, {contact_id}, 0)): {wrench}")
        else:
            print(f"  WARNING: No contact wrench initialized for ContactWrenchKey({link.id()}, {contact_id}, 0)!")


Contact Point Initialization Check:

Contact on left_ankle_roll_link (contact_id=0):
  Link COM pose at t=0: [0.02650267 0.11850645 0.035     ]
  Contact offset in link frame: [-0.05   0.025 -0.035]
  Contact point in world: [-2.34973261e-02  1.43506455e-01 -2.11806858e-09]
  Expected z=0, actual z=-0.000000
  Contact wrench (ContactWrenchKey(6, 0, 0)): [ 3.47043365e-06 -1.31727852e-06  1.12547458e-06 -9.18577739e-06
 -1.39916895e-05  4.94641317e-06]

Contact on left_ankle_roll_link (contact_id=1):
  Link COM pose at t=0: [0.02650267 0.11850645 0.035     ]
  Contact offset in link frame: [-0.05  -0.025 -0.035]
  Contact point in world: [-2.34973261e-02  9.35064550e-02 -2.11806858e-09]
  Expected z=0, actual z=-0.000000
  Contact wrench (ContactWrenchKey(6, 1, 0)): [-2.08574473e-05 -8.53587400e-06 -6.96740937e-06 -8.89325432e-06
 -9.93780777e-07  2.61910447e-06]

Contact on left_ankle_roll_link (contact_id=2):
  Link COM pose at t=0: [0.02650267 0.11850645 0.035     ]
  Contact offset 

## 7.2 Run Optimization

In [20]:
print("\nStarting optimization...")

# Optimize using Levenberg-Marquardt
params = gtsam.LevenbergMarquardtParams()
params.setVerbosityLM("SUMMARY")
params.setAbsoluteErrorTol(1e-6)
params.setRelativeErrorTol(1e-6)
params.setMaxIterations(300)

optimizer = gtsam.LevenbergMarquardtOptimizer(graph, init_values, params)
result_bal = optimizer.optimize()

print(f"\nOptimization complete!")
print(f"Final error: {graph.error(result_bal):.6e}")
print(f"Result contains {result_bal.size()} values")


Starting optimization...

Optimization complete!
Initial error: 1.99346e+14, values: 65618
iter      cost      cost_change    lambda  success iter_time
   0       200369        2e+14      1e-05      1        1.2
   1      7.1e+03      1.9e+05      1e-06      1        1.1
   2      5.3e+02      6.6e+03      1e-07      1        1.2
   3          inf            0      1e-08      0       0.27
   3      5.2e+02           10      1e-07      1        1.3
   4          inf            0      1e-08      0       0.27
   4      5.2e+02      1.1e-06      1e-07      1        1.2
Final error: 5.207333e+02
Result contains 65618 values


In [ ]:
# Analyze factor contributions to FINAL error (after optimization)
print("Analyzing factor contributions to FINAL error (after optimization)...")
print(f"Final total error: {graph.error(result_bal):.6e}\n")

# Get all factors and their errors on optimized values
factor_errors = []
factor_types = []

for i in range(graph.size()):
    factor = graph.at(i)
    error = factor.error(result_bal)
    factor_errors.append(error)
    
    # Get the actual factor type name using Python's type system
    factor_type = type(factor).__name__
    factor_types.append(factor_type)

# Convert to numpy arrays
factor_errors = np.array(factor_errors)
factor_types = np.array(factor_types)

# Group by factor type
unique_types = np.unique(factor_types)
type_errors = {}
type_counts = {}

for ftype in unique_types:
    mask = factor_types == ftype
    type_errors[ftype] = np.sum(factor_errors[mask])
    type_counts[ftype] = np.sum(mask)

# Sort by error contribution
sorted_types = sorted(type_errors.keys(), key=lambda x: type_errors[x], reverse=True)

print(f"Factor Error Analysis (Optimized Values):")
print(f"Total factors: {len(factor_errors)}")
print(f"Total error: {np.sum(factor_errors):.6e}")
print(f"\nError by factor type:")
for ftype in sorted_types:
    print(f"  {ftype:40s}: {type_errors[ftype]:12.6e} ({type_counts[ftype]:4d} factors, avg: {type_errors[ftype]/type_counts[ftype]:.6e})")

# Create visualization with Plotly
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Total Error by Factor Type (Optimized)',
        'Number of Factors by Type',
        'Average Error per Factor by Type (Optimized)',
        'Error Distribution (Log Scale)'
    ),
    specs=[[{"type": "bar"}, {"type": "bar"}],
           [{"type": "bar"}, {"type": "histogram"}]]
)

# Plot 1: Total error by factor type
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#bcbd22', '#17becf']
fig.add_trace(
    go.Bar(x=sorted_types, y=[type_errors[t] for t in sorted_types],
           marker_color=colors[:len(sorted_types)],
           name='Total Error'),
    row=1, col=1
)

# Plot 2: Number of factors by type
fig.add_trace(
    go.Bar(x=sorted_types, y=[type_counts[t] for t in sorted_types],
           marker_color=colors[:len(sorted_types)],
           name='Count'),
    row=1, col=2
)

# Plot 3: Average error per factor by type
fig.add_trace(
    go.Bar(x=sorted_types, y=[type_errors[t]/type_counts[t] for t in sorted_types],
           marker_color=colors[:len(sorted_types)],
           name='Avg Error'),
    row=2, col=1
)

# Plot 4: Error distribution histogram (log scale)
fig.add_trace(
    go.Histogram(x=np.log10(factor_errors[factor_errors > 0]),
                 nbinsx=50,
                 marker_color='#1f77b4',
                 name='Error Distribution'),
    row=2, col=2
)

# Update layout
fig.update_xaxes(tickangle=45, row=1, col=1)
fig.update_xaxes(tickangle=45, row=1, col=2)
fig.update_xaxes(tickangle=45, row=2, col=1)
fig.update_xaxes(title_text="Log10(Error)", row=2, col=2)

fig.update_yaxes(type="log", title_text="Total Error", row=1, col=1)
fig.update_yaxes(title_text="Number of Factors", row=1, col=2)
fig.update_yaxes(type="log", title_text="Average Error", row=2, col=1)
fig.update_yaxes(title_text="Count", row=2, col=2)

fig.update_layout(
    height=800,
    title_text="Factor Error Analysis (Optimized Values - After Optimization)",
    showlegend=False
)

fig.show()

## 7.3 Analyze Final Error After Optimization

In [ ]:
# Print top factors with largest FINAL errors (after optimization)
number_of_factors = 30
print("\n" + "="*80)
print(f"TOP {number_of_factors} FACTORS WITH LARGEST FINAL ERRORS (AFTER OPTIMIZATION)")
print("="*80)

# Create list of (error, factor_type, factor_index) tuples
factor_info = [(factor_errors[i], factor_types[i], i) for i in range(len(factor_errors))]

# Sort by error in descending order
factor_info_sorted = sorted(factor_info, key=lambda x: x[0], reverse=True)

# Print top 15
print(f"\n{'Rank':<6} {'Error':<15} {'Factor Type':<40} {'Factor Index':<12}")
print("-" * 80)
for rank, (error, ftype, idx) in enumerate(factor_info_sorted[:number_of_factors], 1):
    # Get the keys associated with this factor
    factor = graph.at(idx)
    keys = factor.keys()
    key_str = ", ".join([gtd.GTDKeyFormatter(k) for k in keys])
    
    print(f"{rank:<6} {error:<15.6e} {ftype:<40} {idx:<12}")
    print(f"       Keys: {key_str}")
    print()

print("="*80)

In [ ]:
gtd.GTDKeyFormatter(27583465385885697)

## 7.4 Visualize Optimized Torques

Visualize the optimized joint torques across all timesteps to verify the control effort.

In [ ]:
# Extract torques for all joints across the trajectory
time_steps = np.arange(balance_steps + 1) * balance_dt
torques = np.zeros((num_joints, balance_steps + 1))

# Extract torque values from the optimized result
for t in range(balance_steps + 1):
    for j in range(num_joints):
        torque_key = gtd.TorqueKey(j, t)
        if result_bal.exists(torque_key):
            torques[j, t] = result_bal.atDouble(torque_key)

# Get joint names for plotting
joint_names = [joint.name() for joint in robot.joints()]

# Create subplots for each joint
num_rows = (num_joints + 2) // 3  # 3 columns
num_cols = min(3, num_joints)

fig = make_subplots(
    rows=num_rows, cols=num_cols,
    subplot_titles=joint_names,
    vertical_spacing=0.08,
    horizontal_spacing=0.08,
    shared_xaxes=True
)

# Plot torques for each joint
for j in range(num_joints):
    row = (j // 3) + 1
    col = (j % 3) + 1
    
    fig.add_trace(
        go.Scatter(
            x=time_steps,
            y=torques[j],
            mode='lines',
            name=joint_names[j],
            line=dict(width=2),
            showlegend=False
        ),
        row=row, col=col
    )
    
    # Add zero reference line
    fig.add_hline(
        y=0, 
        line_dash="dash", 
        line_color="gray", 
        opacity=0.5, 
        row=row, col=col
    )
    
    # Update y-axis label
    fig.update_yaxes(title_text="Torque (Nm)", row=row, col=col)

# Update x-axis labels (only for bottom row)
for col in range(1, num_cols + 1):
    fig.update_xaxes(title_text="Time (s)", row=num_rows, col=col)

fig.update_layout(
    height=300 * num_rows,
    title_text="Optimized Joint Torques Over Time",
    showlegend=False
)

fig.show()

# Print torque statistics
print("\n=== Joint Torque Statistics ===")
print(f"{'Joint Name':<30} {'Mean (Nm)':<12} {'Max (Nm)':<12} {'Min (Nm)':<12} {'Std (Nm)':<12} {'RMS (Nm)':<12}")
print("-" * 100)

for j in range(num_joints):
    mean_torque = np.mean(torques[j])
    max_torque = np.max(np.abs(torques[j]))
    min_torque = np.min(torques[j])
    std_torque = np.std(torques[j])
    rms_torque = np.sqrt(np.mean(torques[j]**2))
    
    print(f"{joint_names[j]:<30} {mean_torque:>11.4f} {max_torque:>11.4f} {min_torque:>11.4f} {std_torque:>11.4f} {rms_torque:>11.4f}")

print("\n=== Overall Statistics ===")
print(f"Total RMS torque across all joints: {np.sqrt(np.mean(torques**2)):.4f} Nm")
print(f"Maximum absolute torque: {np.max(np.abs(torques)):.4f} Nm")
print(f"Average torque magnitude: {np.mean(np.abs(torques)):.4f} Nm")

## 7.5 Visualize Contact Point Heights

Check that contact points remain on the ground throughout the trajectory.

In [21]:
# Extract and visualize contact point heights over time
time_steps = np.arange(balance_steps + 1) * balance_dt

# Create figure for contact point heights
fig = make_subplots(
    rows=len(contact_points), cols=1,
    subplot_titles=[f"Contact Point: {cp.link.name()}" for cp in contact_points],
    vertical_spacing=0.1,
    shared_xaxes=True
)

contact_heights = {i: [] for i in range(len(contact_points))}

# Calculate contact point heights for each timestep
for t in range(balance_steps + 1):
    for i, point_on_link in enumerate(contact_points):
        link_id = point_on_link.link.id()
        # Get link pose at timestep t
        link_pose = gtd.Pose(result_bal, link_id, t)
        # Transform contact point to world frame
        contact_point_world = link_pose.transformFrom(point_on_link.point)
        # Get z-coordinate (height)
        contact_heights[i].append(contact_point_world[2])

# Plot heights for each contact point
for i, point_on_link in enumerate(contact_points):
    fig.add_trace(
        go.Scatter(
            x=time_steps,
            y=contact_heights[i],
            mode='lines',
            name=point_on_link.link.name(),
            line=dict(width=2, color='blue'),
            showlegend=False
        ),
        row=i+1, col=1
    )
    
    # Add ground plane reference
    fig.add_hline(
        y=ground_plane_height, 
        line_dash="dash", 
        line_color="green", 
        opacity=0.7, 
        row=i+1, col=1,
        annotation_text="Ground" if i == 0 else None
    )
    
    # Update y-axis
    fig.update_yaxes(title_text="Height (m)", row=i+1, col=1)

# Update layout
fig.update_xaxes(title_text="Time (s)", row=len(contact_points), col=1)
fig.update_layout(
    height=300*len(contact_points),
    title_text="Contact Point Heights Over Time",
    showlegend=False
)

fig.show()

# Print statistics
print("\n=== Contact Point Height Statistics ===")
for i, point_on_link in enumerate(contact_points):
    heights = np.array(contact_heights[i])
    mean_height = np.mean(heights)
    max_height = np.max(heights)
    min_height = np.min(heights)
    std_height = np.std(heights)
    print(f"{point_on_link.link.name():30s}: Mean={mean_height:.6f}m, "
          f"Min={min_height:.6f}m, Max={max_height:.6f}m, Std={std_height:.6f}m")


=== Contact Point Height Statistics ===
left_ankle_roll_link          : Mean=-0.000002m, Min=-0.000002m, Max=-0.000000m, Std=0.000000m
left_ankle_roll_link          : Mean=-0.000002m, Min=-0.000002m, Max=-0.000000m, Std=0.000000m
left_ankle_roll_link          : Mean=0.000010m, Min=0.000000m, Max=0.000011m, Std=0.000001m
left_ankle_roll_link          : Mean=0.000010m, Min=0.000000m, Max=0.000011m, Std=0.000001m
right_ankle_roll_link         : Mean=-0.000002m, Min=-0.000002m, Max=-0.000000m, Std=0.000000m
right_ankle_roll_link         : Mean=-0.000002m, Min=-0.000002m, Max=-0.000000m, Std=0.000000m
right_ankle_roll_link         : Mean=0.000010m, Min=0.000000m, Max=0.000011m, Std=0.000001m
right_ankle_roll_link         : Mean=0.000010m, Min=0.000000m, Max=0.000011m, Std=0.000001m


## 7.6. Torso Link Position over time

In [ ]:
# 7.6: Plot torso and pelvis link positions (X, Y, Z) over the optimized trajectory
time_steps = np.arange(balance_steps + 1) * balance_dt

# Find link ids for torso and pelvis (flexible matching)
torso_link_id = None
pelvis_link_id_found = None
for link in robot.links():
    name = link.name().lower()
    if 'torso' in name or 'torso_link' in name:
        torso_link_id = link.id()
    if name == 'pelvis' or 'pelvis' in name:
        pelvis_link_id_found = link.id()

# Fallback to pelvis_link_id variable if set earlier in the notebook
if pelvis_link_id_found is None and 'pelvis_link_id' in globals():
    pelvis_link_id_found = pelvis_link_id

if torso_link_id is None:
    # Try exact 'torso_link' name lookup as last resort
    for link in robot.links():
        if link.name() == 'torso_link':
            torso_link_id = link.id()

print(f'Torso link id: {torso_link_id}, Pelvis link id: {pelvis_link_id_found}')

# Containers for positions
torso_pos = []
pelvis_pos = []

for t in range(balance_steps + 1):
    # Torso pose
    if torso_link_id is not None and result_bal.exists(gtd.PoseKey(torso_link_id, t)):
        p = gtd.Pose(result_bal, torso_link_id, t).translation()
        torso_pos.append(np.array(p))
    else:
        torso_pos.append(np.array([np.nan, np.nan, np.nan]))
    # Pelvis pose
    if pelvis_link_id_found is not None and result_bal.exists(gtd.PoseKey(pelvis_link_id_found, t)):
        p2 = gtd.Pose(result_bal, pelvis_link_id_found, t).translation()
        pelvis_pos.append(np.array(p2))
    else:
        pelvis_pos.append(np.array([np.nan, np.nan, np.nan]))

torso_pos = np.vstack(torso_pos)
pelvis_pos = np.vstack(pelvis_pos)

# Plot with Plotly: two rows (torso, pelvis) each showing X/Y/Z
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=['Torso position (X,Y,Z)', 'Pelvis position (X,Y,Z)'])
colors = {'x':'red','y':'green','z':'blue'}

# Torso traces
fig.add_trace(go.Scatter(x=time_steps, y=torso_pos[:,0], mode='lines', name='Torso X', line=dict(color=colors['x'])), row=1, col=1)
fig.add_trace(go.Scatter(x=time_steps, y=torso_pos[:,1], mode='lines', name='Torso Y', line=dict(color=colors['y'])), row=1, col=1)
fig.add_trace(go.Scatter(x=time_steps, y=torso_pos[:,2], mode='lines', name='Torso Z', line=dict(color=colors['z'])), row=1, col=1)

# Pelvis traces
fig.add_trace(go.Scatter(x=time_steps, y=pelvis_pos[:,0], mode='lines', name='Pelvis X', line=dict(color=colors['x'], dash='dash')), row=2, col=1)
fig.add_trace(go.Scatter(x=time_steps, y=pelvis_pos[:,1], mode='lines', name='Pelvis Y', line=dict(color=colors['y'], dash='dash')), row=2, col=1)
fig.add_trace(go.Scatter(x=time_steps, y=pelvis_pos[:,2], mode='lines', name='Pelvis Z', line=dict(color=colors['z'], dash='dash')), row=2, col=1)

fig.update_xaxes(title_text='Time (s)', row=2, col=1)
fig.update_yaxes(title_text='Position (m)', row=1, col=1)
fig.update_yaxes(title_text='Position (m)', row=2, col=1)
fig.update_layout(height=600, width=900, title_text='Torso and Pelvis Positions over Optimized Trajectory')
fig.show()

# Print statistics per axis for both links
def stats(arr):
    return np.nanmean(arr), np.nanmin(arr), np.nanmax(arr), np.nanstd(arr)

## 8. Linearize and Create LQR Policy (Bayes Net)

In [22]:
print("Linearizing factor graph around optimized trajectory...")

# Linearize the nonlinear factor graph
gaussian_graph_bal = graph.linearize(result_bal)

print(f"Linearized graph contains {gaussian_graph_bal.size()} Gaussian factors")

# Create elimination ordering (backward in time)
# Controls (torques) first, then states (angles, velocities, accelerations)
print("Creating backward elimination ordering...")
ordering_bal = gtsam.Ordering()
contact_link_ids = [cp.link.id() for cp in contact_points]

num_links = robot.numLinks()

# Add variables from t = balance_steps down to t = 0
for t in reversed(range(balance_steps + 1)):
    # Torques (controls) - eliminate first
    for j in range(num_joints):
        key = gtd.TorqueKey(j, t)
        if result_bal.exists(key):
            ordering_bal.push_back(key)
    
    # Contact wrenches (state) - eliminate after controls
    # Process contact points grouped by link, matching C++ implementation
    for link in robot.links():
        link_id = link.id()
        contact_id = 0  # Track unique contact ID per link
        
        for cp in contact_points:
            if cp.link.id() != link_id:
                continue
            # Use unique contact ID for each contact point on this link
            key = gtd.ContactWrenchKey(link_id, contact_id, t)
            if result_bal.exists(key):
                ordering_bal.push_back(key)
            contact_id += 1
    
    # Joint angles (state)
    for j in range(num_joints):
        key = gtd.JointAngleKey(j, t)
        if result_bal.exists(key): 
            ordering_bal.push_back(key)

    # Link poses (state)
    for i in range(num_links):
        key = gtd.PoseKey(i, t)
        if result_bal.exists(key): 
            ordering_bal.push_back(key)
    
    # Accelerations (state)
    for j in range(num_joints):
        key = gtd.JointAccelKey(j, t)
        if result_bal.exists(key): 
            ordering_bal.push_back(key)
    for i in range(num_links):
        key = gtd.TwistAccelKey(i, t)
        if result_bal.exists(key): 
            ordering_bal.push_back(key)

    # Wrenches (state)
    for i in range(num_links):
        for j in range(num_joints):
            key = gtd.WrenchKey(i, j, t)
            if result_bal.exists(key): 
                ordering_bal.push_back(key)

    # Velocities (state)
    for j in range(num_joints):
        key = gtd.JointVelKey(j, t)
        if result_bal.exists(key): 
            ordering_bal.push_back(key)
    for i in range(num_links):
        key = gtd.TwistKey(i, t)
        if result_bal.exists(key): 
            ordering_bal.push_back(key)

print(f"Elimination ordering created with {ordering_bal.size()} keys")

# Verify ordering
if ordering_bal.size() != result_bal.size():
    print(f"WARNING: Ordering size ({ordering_bal.size()}) != result size ({result_bal.size()})")
else:
    print("Ordering size matches result size ✓")

Linearizing factor graph around optimized trajectory...
Linearized graph contains 95950 Gaussian factors
Creating backward elimination ordering...
Elimination ordering created with 65618 keys
Ordering size matches result size ✓


In [23]:
# Eliminate to get the Bayes Net (LQR policy)
print("\nPerforming sequential elimination...")
policy_bayes_net_bal = gaussian_graph_bal.eliminateSequential(ordering_bal)

print(f"Bayes Net created with {policy_bayes_net_bal.size()} conditionals")
print("LQR policy ready!")


Performing sequential elimination...


RuntimeError: 
Indeterminant linear system detected while working near variable
18865429099315500 (Symbol: 18865429099315500).

Thrown when a linear system is ill-posed.  The most common cause for this
error is having underconstrained variables.  Mathematically, the system is
underdetermined.  See the GTSAM Doxygen documentation at
http://borg.cc.gatech.edu/ on gtsam::IndeterminantLinearSystemException for
more information.

In [ ]:
gtd.GTDKeyFormatter(24233231981216040)

## 9. MuJoCo Simulation Setup

In [ ]:
# Initialize MuJoCo model and data
model = mujoco.MjModel.from_xml_path(MJCF_PATH)
data = mujoco.MjData(model)

print(f"MuJoCo model loaded")
print(f"Number of MuJoCo qpos: {model.nq}")
print(f"Number of MuJoCo qvel: {model.nv}")
print(f"Number of MuJoCo actuators: {model.nu}")

# Create mapping from GTDynamics joint order to MuJoCo joint indices
# MuJoCo uses different ordering than URDF
gtd_to_mujoco_pos = {}  # For qpos addresses
gtd_to_mujoco_vel = {}  # For qvel addresses
mujoco_joint_names = []

for i in range(model.njnt):
    joint_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, i)
    if joint_name:
        
        mujoco_joint_names.append(joint_name)

print(f"\nMuJoCo joint names ({len(mujoco_joint_names)}):")
for i, name in enumerate(mujoco_joint_names):
    print(f"  {i}: {name}")

# Map GTDynamics joints to MuJoCo joint addresses
for joint in robot.joints():
    gtd_idx = joint.id()
    gtd_joint_name = joint.name()
    # Find corresponding MuJoCo joint
    for mj_idx, mj_name in enumerate(mujoco_joint_names):
        if gtd_joint_name == mj_name:
            # Get the qpos and qvel addresses for this joint
            joint_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, mj_name)
            qpos_addr = model.jnt_qposadr[joint_id]
            qvel_addr = model.jnt_dofadr[joint_id]  # Use dofadr for velocity
            gtd_to_mujoco_pos[gtd_idx] = qpos_addr
            gtd_to_mujoco_vel[gtd_idx] = qvel_addr
            break

print(f"\nMapped {len(gtd_to_mujoco_pos)} joints from GTDynamics to MuJoCo")
print(f"Position mapping size: {len(gtd_to_mujoco_pos)}")
print(gtd_to_mujoco_pos)
print(f"Velocity mapping size: {len(gtd_to_mujoco_vel)}")
print(gtd_to_mujoco_vel)

# Create mapping from GTDynamics link IDs to MuJoCo body IDs
gtd_to_mujoco_body = {}
for link in robot.links():
    link_name = link.name()
    gtd_link_id = link.id()
    # Find corresponding MuJoCo body
    try:
        mj_body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, link_name)
        gtd_to_mujoco_body[gtd_link_id] = mj_body_id
    except:
        print(f"Warning: Could not find MuJoCo body for link '{link_name}'")

print(f"\nMapped {len(gtd_to_mujoco_body)} links from GTDynamics to MuJoCo bodies")
print(gtd_to_mujoco_body)

gtd_joint_to_mujoco_actuator = {}
for i in range(model.nu):
    actuator_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, i)
    # Assume actuator name matches joint name
    for joint in robot.joints():
        if joint.name() == actuator_name:
            gtd_joint_to_mujoco_actuator[joint.id()] = i
            break

print(f"\nMapped {len(gtd_joint_to_mujoco_actuator)} joints from GTDynamics to MuJoCo actuators")
print(gtd_joint_to_mujoco_actuator)


## Simulation without LQR

In [ ]:
# Simulation with no torques
# Reset MuJoCo data to a clean state
mujoco.mj_resetData(model, data)

physics_dt = 0.01  # physics internal step
model.opt.timestep = physics_dt

# Rendering parameters (small video capture)
VIDEO_WIDTH = 800
VIDEO_HEIGHT = 600
render_freq = 30
frames = []

# Run simulation and apply u_ref each control step
with mujoco.Renderer(model, width=VIDEO_WIDTH, height=VIDEO_HEIGHT) as renderer:
    for t in range(300):
        current_time = t * physics_dt

        # Print Contact points form mujoco data
        print(f"Time {current_time:.3f}s - Contact Points:")
        for i in range(data.ncon):
            con = data.contact[i]
            # You can check which geoms are involved
            # if (con.geom1 == geom1_id and con.geom2 == geom2_id):
            contact_pos = con.pos
            print(f"    Contact {i}: position = {contact_pos}")


        mujoco.mj_step(model, data)
        renderer.update_scene(data)
        frames.append(renderer.render())

        if (t + 1) % 10 == 0:
            print(f"Progress: {t+1}/{300} control steps")

print('Simulation complete')
# Display video frames (if running in notebook environment)
try:
    media.show_video(frames, fps=render_freq, width=VIDEO_WIDTH, height=VIDEO_HEIGHT)
except Exception as e:
    print('Could not show video inline:', e)

In [ ]:
# Simulation applying optimized torques directly (NO LQR)
# This cell runs MuJoCo and for each control timestep applies the optimized torque u_ref from the trajectory
# It fills the same tracking_data structure used by the later plotting cells so those plots work unchanged.

# Reset MuJoCo data to a clean state
mujoco.mj_resetData(model, data)

# Set initial joint angles from the optimization initial guess (or zeros)
for gtd_idx in range(num_joints):
    if gtd_idx in gtd_to_mujoco_pos:
        mj_qpos_addr = gtd_to_mujoco_pos[gtd_idx]
        data.qpos[mj_qpos_addr] = initial_joint_angles[gtd_idx]

# Control / physics timestep setup (reuse the same values as TVLQR cell)
control_dt = balance_dt
physics_dt = 0.01  # physics internal step
model.opt.timestep = physics_dt
sim_substeps = int(max(1, round(control_dt / physics_dt)))

print(f"Running open-loop MuJoCo sim applying optimized torques for {balance_steps} control steps (substeps={sim_substeps})...")

# Rendering parameters (small video capture)
VIDEO_WIDTH = 800
VIDEO_HEIGHT = 600
render_freq = 30
frames = []

# Prepare tracking_data (same structure used later by plotting cells)
tracking_data = {
    'time': [],
    'joint_angle_deviations': [[] for _ in range(num_joints)],
    'joint_vel_deviations': [[] for _ in range(num_joints)],
    'torque_corrections': [[] for _ in range(num_joints)],  # zeros for open-loop
    'torque_references': [[] for _ in range(num_joints)],
    'torque_totals': [[] for _ in range(num_joints)],
    'base_position': [],
    'base_orientation': [],
    'planned_joint_angles': [[] for _ in range(num_joints)],
    'actual_joint_angles': [[] for _ in range(num_joints)],
    'planned_base_position': [],
    'actual_base_position': []
}

# Run simulation and apply u_ref each control step
with mujoco.Renderer(model, width=VIDEO_WIDTH, height=VIDEO_HEIGHT) as renderer:
    for t in range(balance_steps):
        current_time = t * balance_dt

        # Read current actual joint states from MuJoCo
        q_actual = np.zeros(num_joints)
        v_actual = np.zeros(num_joints)
        for gtd_idx in range(num_joints):
            if gtd_idx in gtd_to_mujoco_pos:
                mj_qpos_addr = gtd_to_mujoco_pos[gtd_idx]
                mj_qvel_addr = gtd_to_mujoco_vel.get(gtd_idx, None)
                q_actual[gtd_idx] = data.qpos[mj_qpos_addr]
                if mj_qvel_addr is not None:
                    v_actual[gtd_idx] = data.qvel[mj_qvel_addr]

        # Get reference (planned) state from optimized trajectory
        q_ref = np.array([gtd.JointAngle(result_bal, j, t) for j in range(num_joints)])
        v_ref = np.array([gtd.JointVel(result_bal, j, t) for j in range(num_joints)])
        u_ref = np.array([gtd.Torque(result_bal, j, t) for j in range(num_joints)])

        # Print Contact points form mujoco data
        print(f"Time {current_time:.3f}s - Contact Points:")
        for i in range(data.ncon):
            con = data.contact[i]
            # You can check which geoms are involved
            # if (con.geom1 == geom1_id and con.geom2 == geom2_id):
            contact_pos = con.pos
            print(f"    Contact {i}: position = {contact_pos}")

        # Open-loop: no correction (du = 0)
        du = np.zeros(num_joints)
        u_total = u_ref.copy()

        # Apply to MuJoCo actuators (naive mapping: use same index if actuator exists)
        for j in range(num_joints):
            if j < model.nu:  # actuator exists
                # Clip/convert if needed here (left as-is)
                data.ctrl[j] = float(u_total[j])

        # Record tracking data
        tracking_data['time'].append(current_time)
        base_pos = data.qpos[:3].copy()
        base_quat = data.qpos[3:7].copy()
        tracking_data['base_position'].append(base_pos)
        tracking_data['base_orientation'].append(base_quat)
        tracking_data['planned_base_position'].append(gtd.Pose(result_bal, pelvis_link_id, t).translation())
        tracking_data['actual_base_position'].append(base_pos)

        for j in range(num_joints):
            tracking_data['joint_angle_deviations'][j].append(q_actual[j] - q_ref[j])
            tracking_data['joint_vel_deviations'][j].append(v_actual[j] - v_ref[j])
            tracking_data['torque_corrections'][j].append(0.0)
            tracking_data['torque_references'][j].append(u_ref[j])
            tracking_data['torque_totals'][j].append(u_total[j])
            tracking_data['planned_joint_angles'][j].append(q_ref[j])
            tracking_data['actual_joint_angles'][j].append(q_actual[j])

        # Step physics for sim_substeps and render once per control step
        for _ in range(sim_substeps):
            mujoco.mj_step(model, data)

        renderer.update_scene(data)
        frames.append(renderer.render())

        if (t + 1) % 10 == 0:
            print(f"Progress (open-loop): {t+1}/{balance_steps} control steps")

print('Open-loop simulation complete')
# Display video frames (if running in notebook environment)
try:
    media.show_video(frames, fps=render_freq, width=VIDEO_WIDTH, height=VIDEO_HEIGHT)
except Exception as e:
    print('Could not show video inline:', e)

## 10. Run TVLQR Feedback Control Simulation

In [ ]:
help(data)

In [ ]:
# Reset MuJoCo simulation to clean state
mujoco.mj_resetData(model, data)
mujoco.mj_forward(model, data)

#data.qpos[:3] = np.array([0.0, 0.0, 0.793])  # Base position
#data.qpos[3:7] = np.array([1.0, 0.0, 0.0, 0.0])  # Base quaternion (identity: w=1, x=0, y=0, z=0)
orientation_flat = data.ximat[gtd_to_mujoco_body[pelvis_link_id]]


actual_base_pos = data.xipos[gtd_to_mujoco_body[pelvis_link_id]]  # x, y, z
actual_base_rot = data.ximat[gtd_to_mujoco_body[pelvis_link_id]] # flatten 3x3 matrix
actual_base_rot = gtsam.Rot3(actual_base_rot.reshape(3,3))
actual_base_pose = gtsam.Pose3(actual_base_rot, actual_base_pos)

print(actual_base_pose)

In [ ]:
# Reset MuJoCo simulation to clean state
mujoco.mj_resetData(model, data)
mujoco.mj_forward(model, data)


# Define the two different timesteps
control_dt = balance_dt   
physics_dt = 0.01 
model.opt.timestep = physics_dt
sim_substeps = int(control_dt / physics_dt)

print("Starting MuJoCo simulation with LQR feedback control...")

# Rendering parameters
VIDEO_WIDTH = 800
VIDEO_HEIGHT = 600
render_freq = 30  # fps
frames = []

# Data storage for tracking deviations and control
tracking_data = {
    'time': [],
    'joint_angle_deviations': [[] for _ in range(num_joints)],
    'joint_vel_deviations': [[] for _ in range(num_joints)],
    'torque_corrections': [[] for _ in range(num_joints)],
    'torque_references': [[] for _ in range(num_joints)],
    'torque_totals': [[] for _ in range(num_joints)],
    'base_position': [],
    'base_orientation': [],
    'planned_joint_angles': [[] for _ in range(num_joints)],
    'actual_joint_angles': [[] for _ in range(num_joints)],
    'planned_base_position': [],
    'actual_base_position': []
}

with mujoco.Renderer(model, width=VIDEO_WIDTH, height=VIDEO_HEIGHT) as renderer:
    for t in range(balance_steps):
        current_time = t * balance_dt
        
        # Get actual state from MuJoCo
        q_actual = np.zeros(num_joints)
        v_actual = np.zeros(num_joints)
        
        for joint in robot.joints():
            gtd_idx = joint.id()
            if gtd_idx in gtd_to_mujoco_pos:
                mj_qpos_addr = gtd_to_mujoco_pos[gtd_idx]
                mj_qvel_addr = gtd_to_mujoco_vel[gtd_idx]
                q_actual[gtd_idx] = data.qpos[mj_qpos_addr]
                v_actual[gtd_idx] = data.qvel[mj_qvel_addr]
        
        # Get reference state from optimized trajectory
        q_ref = np.array([gtd.JointAngle(result_bal, j.id(), t) for j in robot.joints()])
        v_ref = np.array([gtd.JointVel(result_bal, j.id(), t) for j in robot.joints()])
        u_ref = np.array([gtd.Torque(result_bal, j.id(), t) for j in robot.joints()])
        
        # Get planned base position and twist
        planned_base_pose = gtd.Pose(result_bal, pelvis_link_id, t)
        planned_base_pos = planned_base_pose.translation()
        planned_base_twist = gtd.Twist(result_bal, pelvis_link_id, t)
        
        # Get actual base pose and twist from MuJoCo
        actual_base_pos = data.xipos[gtd_to_mujoco_body[pelvis_link_id]]  # x, y, z
        actual_base_rot = data.ximat[gtd_to_mujoco_body[pelvis_link_id]] # flatten 3x3 matrix
        actual_base_rot = gtsam.Rot3(actual_base_rot.reshape(3,3))
        actual_base_pose = gtsam.Pose3(actual_base_rot, actual_base_pos)
        
        # Get MuJoCo's twist components
        actual_base_lin_vel_global = data.qvel[:3]
        actual_base_ang_vel_local = data.qvel[3:6]
        
        # Convert local angular velocity to the global frame
        #    We use the rotation matrix R_w_b (world_R_body)
        R_w_b = actual_base_rot.matrix()  # Get the 3x3 numpy rotation matrix
        actual_base_ang_vel_global = R_w_b @ actual_base_ang_vel_local
        
        # Assemble the actual twist in the global frame
        #    This follows [ang, lin] convention
        actual_twist_gtd_convention = np.concatenate([actual_base_ang_vel_global, actual_base_lin_vel_global])

        # Compute state deviations
        dx = gtsam.VectorValues()
        
        # Add joint state deviations
        q_deviations = np.zeros(num_joints)
        v_deviations = np.zeros(num_joints)

        for joint in robot.joints():
            gtd_idx = joint.id()

            q_ref = gtd.JointAngle(result_bal, gtd_idx, t)
            q_act = q_actual[gtd_idx]
            q_deviations[gtd_idx] = q_act - q_ref

            v_ref = gtd.JointVel(result_bal, gtd_idx, t)
            v_act = v_actual[gtd_idx]
            v_deviations[gtd_idx] = v_act - v_ref

            dx.insert(gtd.JointAngleKey(gtd_idx, t), np.array([q_act - q_ref]))
            dx.insert(gtd.JointVelKey(gtd_idx, t), np.array([v_act - v_ref]))
        
        base_pose_error = actual_base_pose.between(planned_base_pose)
        base_pose_error_vector = gtsam.Pose3.Logmap(base_pose_error)
        print("="*40)
        print(f"Timestep t={t}, current_time={current_time:.3f}s")

        print(f"Planned base pose at t={t}: {planned_base_pose}")
        print(f"Actual base pose at t={t}: {actual_base_pose}")
        print(f"Base pose error at t={t}: {base_pose_error}")
        print(f"Base pose error vector(logmap) at t={t}: {base_pose_error_vector}")


        dx.insert(gtd.PoseKey(pelvis_link_id, t), base_pose_error_vector)
        base_twist_error = actual_twist_gtd_convention - planned_base_twist
        dx.insert(gtd.TwistKey(pelvis_link_id, t), base_twist_error)

        # Filter Bayes net to remove provided state variables
        provided_keys = set()
        for joint in robot.joints():
            gtd_idx = joint.id()
            provided_keys.add(gtd.JointAngleKey(gtd_idx, t))
            provided_keys.add(gtd.JointVelKey(gtd_idx, t))
        
        # Add ONLY base link keys to the provided set
        provided_keys.add(gtd.PoseKey(pelvis_link_id, t))
        provided_keys.add(gtd.TwistKey(pelvis_link_id, t))
        
        filtered_bayes_net = gtsam.GaussianBayesNet()
        for i in range(policy_bayes_net_bal.size()):
            conditional = policy_bayes_net_bal.at(i)
            frontal_key = conditional.firstFrontalKey()
            if frontal_key not in provided_keys:
                filtered_bayes_net.push_back(conditional)
        
        # Compute optimal corrections using LQR policy
        optimal_corrections = filtered_bayes_net.optimize(dx)
        
        # Extract torque corrections
        du = np.zeros(num_joints)
        for joint in robot.joints():
            gtd_idx = joint.id()
            du[gtd_idx] = optimal_corrections.at(gtd.TorqueKey(gtd_idx, t))[0]
        
        u_total = np.zeros(num_joints)
        # Set MuJoCo control
        for joint in robot.joints():
            gtd_idx = joint.id()
            if gtd_idx in gtd_joint_to_mujoco_actuator:
                actuator_idx = gtd_joint_to_mujoco_actuator[gtd_idx]
                if actuator_idx < model.nu:  # Check if actuator exists
                    u_ref = gtd.Torque(result_bal, gtd_idx, t)
                    u_final = u_ref + du[gtd_idx]
                    u_total[gtd_idx] = u_final

                    data.ctrl[actuator_idx] = u_final
        print(f"Applied torques at t={t}:")
        for joint in robot.joints():
            gtd_idx = joint.id()
            print(f"  Joint {joint.name()} (gtd_idx={gtd_idx}): u_ref={gtd.Torque(result_bal, gtd_idx, t):.4f}, du={du[gtd_idx]:.4f}, u_total={u_total[gtd_idx]:.4f}")
        
        # Store tracking data
        tracking_data['time'].append(current_time)
        base_pos = data.qpos[:3].copy()
        base_quat = data.qpos[3:7].copy()
        tracking_data['base_position'].append(base_pos)
        tracking_data['base_orientation'].append(base_quat)
        tracking_data['planned_base_position'].append(planned_base_pos)
        tracking_data['actual_base_position'].append(actual_base_pos)
        
        for joint in robot.joints():
            gtd_idx = joint.id()
            tracking_data['joint_angle_deviations'][gtd_idx].append(q_deviations[gtd_idx])
            tracking_data['joint_vel_deviations'][gtd_idx].append(v_deviations[gtd_idx])
            tracking_data['torque_corrections'][gtd_idx].append(du[gtd_idx])
            tracking_data['torque_references'][gtd_idx].append(gtd.Torque(result_bal, gtd_idx, t))
            tracking_data['torque_totals'][gtd_idx].append(u_total[gtd_idx])
            tracking_data['planned_joint_angles'][gtd_idx].append(gtd.JointAngle(result_bal, gtd_idx, t))
            tracking_data['actual_joint_angles'][gtd_idx].append(q_actual[gtd_idx])
        
        # Simulate and render frames for this control timestep
        for _ in range(sim_substeps):
            mujoco.mj_step(model, data)

        # Render frame at the end of the control step
        renderer.update_scene(data)
        frames.append(renderer.render())
        
        if (t + 1) % 10 == 0:
            print(f"Progress: {t+1}/{balance_steps} steps")

        if t >= 26:
            break

print("Simulation complete!")

# Show the video
media.show_video(frames, fps=render_freq, width=VIDEO_WIDTH, height=VIDEO_HEIGHT)

## 11. Visualize Joint Deviations (All Joints)

In [ ]:
# Prepare data
time = tracking_data['time']
joint_names = [joint.name() for joint in robot.joints()]

# Create figure with subplots for joint angle deviations
fig = make_subplots(
    rows=num_joints, cols=1,
    subplot_titles=[f"Joint {i}: {joint_names[i]}" for i in range(num_joints)],
    vertical_spacing=0.02,
    shared_xaxes=True
)

# Add traces for each joint's angle deviation
for j in range(num_joints):
    fig.add_trace(
        go.Scatter(
            x=time,
            y=np.rad2deg(tracking_data['joint_angle_deviations'][j]),
            mode='lines',
            name=joint_names[j],
            line=dict(width=2),
            showlegend=False
        ),
        row=j+1, col=1
    )
    
    # Add zero line
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=j+1, col=1)
    
    # Update y-axis
    fig.update_yaxes(title_text="Deg", row=j+1, col=1, title_font=dict(size=10))

# Update layout
fig.update_xaxes(title_text="Time (s)", row=num_joints, col=1)
fig.update_layout(
    height=300*num_joints,
    title_text="Joint Angle Deviations (Actual - Planned)",
    showlegend=False
)

fig.show()

# Create figure for joint velocity deviations
fig2 = make_subplots(
    rows=num_joints, cols=1,
    subplot_titles=[f"Joint {i}: {joint_names[i]}" for i in range(num_joints)],
    vertical_spacing=0.02,
    shared_xaxes=True
)

# Add traces for each joint's velocity deviation
for j in range(num_joints):
    fig2.add_trace(
        go.Scatter(
            x=time,
            y=np.rad2deg(tracking_data['joint_vel_deviations'][j]),
            mode='lines',
            name=joint_names[j],
            line=dict(width=2, color='red'),
            showlegend=False
        ),
        row=j+1, col=1
    )
    
    # Add zero line
    fig2.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=j+1, col=1)
    
    # Update y-axis
    fig2.update_yaxes(title_text="Deg/s", row=j+1, col=1, title_font=dict(size=10))

# Update layout
fig2.update_xaxes(title_text="Time (s)", row=num_joints, col=1)
fig2.update_layout(
    height=300*num_joints,
    title_text="Joint Velocity Deviations (Actual - Planned)",
    showlegend=False
)

fig2.show()

# Print statistics
print("\n=== Deviation Statistics ===")
for j in range(num_joints):
    max_angle_dev = max(np.abs(tracking_data['joint_angle_deviations'][j]))
    max_vel_dev = max(np.abs(tracking_data['joint_vel_deviations'][j]))
    print(f"{joint_names[j]:30s}: Angle={np.rad2deg(max_angle_dev):8.4f} deg, Vel={np.rad2deg(max_vel_dev):8.4f} deg/s")

## 12. Planned vs Actual Joint Angles

In [ ]:
# Create figure comparing planned vs actual joint angles
fig = make_subplots(
    rows=num_joints, cols=1,
    subplot_titles=[f"Joint {i}: {joint_names[i]}" for i in range(num_joints)],
    vertical_spacing=0.02,
    shared_xaxes=True
)

# Add traces for each joint
for j in range(num_joints):
    # Planned angles
    fig.add_trace(
        go.Scatter(
            x=time,
            y=np.rad2deg(tracking_data['planned_joint_angles'][j]),
            mode='lines',
            name='Planned',
            line=dict(width=2, color='blue', dash='dash'),
            legendgroup='planned',
            showlegend=(j==0)
        ),
        row=j+1, col=1
    )
    
    # Actual angles
    fig.add_trace(
        go.Scatter(
            x=time,
            y=np.rad2deg(tracking_data['actual_joint_angles'][j]),
            mode='lines',
            name='Actual',
            line=dict(width=2, color='red'),
            legendgroup='actual',
            showlegend=(j==0)
        ),
        row=j+1, col=1
    )
    
    # Update y-axis
    fig.update_yaxes(title_text="Deg", row=j+1, col=1, title_font=dict(size=10))

# Update layout
fig.update_xaxes(title_text="Time (s)", row=num_joints, col=1)
fig.update_layout(
    height=300*num_joints,
    title_text="Planned vs Actual Joint Angles",
    showlegend=True,
    legend=dict(x=1.05, y=1, xanchor='left', yanchor='top')
)

fig.show()

## 13. Applied Torques Visualization

In [ ]:
# Visualize applied torques for all joints
fig = make_subplots(
    rows=num_joints, cols=1,
    subplot_titles=[f"Joint {i}: {joint_names[i]}" for i in range(num_joints)],
    vertical_spacing=0.02,
    shared_xaxes=True
)

# Add traces for each joint's torque
for j in range(num_joints):
    # Total torque (reference + correction)
    fig.add_trace(
        go.Scatter(
            x=time,
            y=tracking_data['torque_totals'][j],
            mode='lines',
            name='Total',
            line=dict(width=2, color='black'),
            legendgroup='total',
            showlegend=(j==0)
        ),
        row=j+1, col=1
    )
    
    # Reference torque (from optimization)
    fig.add_trace(
        go.Scatter(
            x=time,
            y=tracking_data['torque_references'][j],
            mode='lines',
            name='Reference',
            line=dict(width=1.5, color='blue', dash='dash'),
            legendgroup='reference',
            showlegend=(j==0)
        ),
        row=j+1, col=1
    )
    
    # Correction torque (from LQR feedback)
    fig.add_trace(
        go.Scatter(
            x=time,
            y=tracking_data['torque_corrections'][j],
            mode='lines',
            name='Correction',
            line=dict(width=1.5, color='red', dash='dot'),
            legendgroup='correction',
            showlegend=(j==0)
        ),
        row=j+1, col=1
    )
    
    # Add zero line
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=j+1, col=1)
    
    # Update y-axis
    fig.update_yaxes(title_text="N·m", row=j+1, col=1, title_font=dict(size=10))

# Update layout
fig.update_xaxes(title_text="Time (s)", row=num_joints, col=1)
fig.update_layout(
    height=300*num_joints,
    title_text="Applied Torques - All Joints (TVLQR Control)",
    showlegend=True,
    legend=dict(x=1.05, y=1, xanchor='left', yanchor='top')
)

fig.show()

# Print torque statistics
print("\n=== Applied Torque Statistics ===")
for j in range(num_joints):
    max_total = max(np.abs(tracking_data['torque_totals'][j]))
    max_ref = max(np.abs(tracking_data['torque_references'][j]))
    max_corr = max(np.abs(tracking_data['torque_corrections'][j]))
    mean_total = np.mean(np.abs(tracking_data['torque_totals'][j]))
    print(f"{joint_names[j]:30s}: Max Total={max_total:8.4f} N·m, Max Ref={max_ref:8.4f} N·m, Max Corr={max_corr:8.4f} N·m, Mean |Total|={mean_total:8.4f} N·m")

In [ ]:
# Plot planned vs actual pelvis base position (X, Y, Z) from simulation tracking_data
import numpy as np
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Ensure tracking_data is present
if 'tracking_data' not in globals():
    print("tracking_data not found. Run the simulation cells to populate tracking_data before plotting.")
else:
    time = np.array(tracking_data.get('time', []))
    planned = np.array(tracking_data.get('planned_base_position', []))
    actual = np.array(tracking_data.get('actual_base_position', []))

    if time.size == 0 or planned.size == 0 or actual.size == 0:
        print("No planned/actual base position data found in tracking_data. Make sure the simulation cells were run and tracking_data was populated.")
    else:
        # Try to reshape to (N,3) if necessary
        try:
            planned = planned.reshape(len(time), 3)
        except Exception:
            planned = np.asarray([np.asarray(p) for p in planned])
        try:
            actual = actual.reshape(len(time), 3)
        except Exception:
            actual = np.asarray([np.asarray(a) for a in actual])

        coords = ['X', 'Y', 'Z']
        fig = make_subplots(rows=3, cols=1, shared_xaxes=True, subplot_titles=[f'Pelvis {c} position' for c in coords])

        for i, c in enumerate(coords):
            fig.add_trace(
                go.Scatter(x=time, y=planned[:, i], mode='lines', name=f'Planned {c}', line=dict(color='blue')),
                row=i+1, col=1
            )
            fig.add_trace(
                go.Scatter(x=time, y=actual[:, i], mode='lines', name=f'Actual {c}', line=dict(color='red')),
                row=i+1, col=1
            )
            fig.update_yaxes(title_text='Position (m)', row=i+1, col=1)

        fig.update_xaxes(title_text='Time (s)', row=3, col=1)
        fig.update_layout(height=800, title_text='Planned vs Actual Pelvis Base Position (X,Y,Z)')
        fig.show()

        # Compute and print RMSE per axis
        rmse = np.sqrt(np.mean((planned - actual)**2, axis=0))
        print(f"RMSE (m) - X: {rmse[0]:.6e}, Y: {rmse[1]:.6e}, Z: {rmse[2]:.6e}")


## 14. Table